# CEDAR + Dataset2 Signature Verification — Colab Training Notebook

Trains the dual-branch ResNet-50 Siamese network on **both datasets combined**, then evaluates on **two independent held-out test sets**.

| | Training | Test (held out) |
|---|---|---|
| **CEDAR** | Writers 1–45 | Writers 46–55 |
| **Dataset2** | Writers 1–650 | Writers 651–686 |

Primary metric reported: **FAR @ 0.5%** — forgeries accepted at the strictest banking threshold.

---

## Estimated time on Colab T4 GPU
| Step | Time |
|---|---|
| Cache CEDAR (2,640 images, first run only) | ~2 min |
| Cache Dataset2 (~13,700 images, first run only) | ~12 min |
| **Training 100 epochs** | **~4–5 hours** |
| Evaluation both test sets | ~8 min |

Cache steps are skipped on re-runs (existing .npy files are not overwritten).

---

## Setup steps

### 1. Enable GPU (do this FIRST)
Runtime → Change runtime type → T4 GPU → Save

### 2. Upload files
Click the folder icon in the sidebar → upload icon, then upload:
- `preprocess.py`, `model.py`, `dataset.py`, `train.py`, `evaluate.py`
- `data.zip` — CEDAR (extracts to `full_org/` and `full_forg/`)
- `data2.zip` — Dataset2 (extracts to `dataset2/001/`, `dataset2/001_forg/`, …)

### 3. Runtime → Run all
Cells run in order. After training, results download automatically.

> Colab disconnects after ~12 hours idle. Download `results.zip` before closing the tab.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'No GPU found.\n'
        'Go to Runtime → Change runtime type → T4 GPU → Save, then re-run.'
    )

props = torch.cuda.get_device_properties(0)
print(f'GPU  : {props.name}')
print(f'VRAM : {props.total_memory / 1e9:.1f} GB')
print(f'CUDA : {torch.version.cuda}')

In [ ]:
# ── Cell 2: Install extra packages ────────────────────────────────────────
# torch + torchvision are pre-installed in Colab; only opencv needs adding.
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'],
    check=True,
)
print('Packages ready.')

In [ ]:
# ── Cell 3: Verify uploads and unzip both datasets ────────────────────────
import os, zipfile, sys
sys.path.insert(0, '.')

# Check all Python files are present
for f in ['preprocess.py', 'model.py', 'dataset.py', 'train.py', 'evaluate.py']:
    if not os.path.exists(f):
        raise FileNotFoundError(f'{f} missing — upload it via the Files panel (Step 2).')
    print(f'  OK  {f}')

# ── CEDAR ─────────────────────────────────────────────────────────────────
if not os.path.exists('data.zip'):
    raise FileNotFoundError('data.zip missing — upload CEDAR archive (Step 2).')
print('\nUnzipping CEDAR (data.zip) ...')
with zipfile.ZipFile('data.zip', 'r') as zf:
    zf.extractall('.')
for folder in ('full_org', 'full_forg'):
    if os.path.isdir(folder):
        print(f'  {folder}/  ({len(os.listdir(folder))} files)')
    else:
        print(f'  WARNING: {folder}/ not found — check data.zip structure')

# ── Dataset2 ──────────────────────────────────────────────────────────────
if not os.path.exists('data2.zip'):
    raise FileNotFoundError(
        'data2.zip missing — upload Dataset2 archive (Step 2).\n'
        'Expected structure inside zip: dataset2/001/, dataset2/001_forg/, ...'
    )
print('\nUnzipping Dataset2 (data2.zip) ...')
with zipfile.ZipFile('data2.zip', 'r') as zf:
    zf.extractall('.')
if os.path.isdir('dataset2'):
    writer_dirs = [
        d for d in os.listdir('dataset2')
        if os.path.isdir(os.path.join('dataset2', d)) and not d.endswith('_forg')
    ]
    print(f'  dataset2/  ({len(writer_dirs)} writer folders found)')
else:
    print('  WARNING: dataset2/ folder not found after unzip.')
    print('  data2.zip must contain a top-level dataset2/ folder.')

print('\nAll files verified.')

In [ ]:
# ── Cell 4: Build image caches ────────────────────────────────────────────
# Preprocesses every image once (binarise / crop / resize to 224×224)
# and saves the result as a .npy file.  Safe to re-run — existing files
# are skipped, so only new images are processed.
from dataset import build_cache, build_cache_dataset2

print('=== CEDAR cache (writers 1-55) ===')
build_cache('.', 'cache')

print('\n=== Dataset2 cache (writers 1-686) ===')
build_cache_dataset2('dataset2', 'cache')

In [ ]:
# ── Cell 5: Train on CEDAR (1-45) + Dataset2 (1-650) combined ─────────────
#
# What's improved vs the previous run:
#
#   ArcFace loss     : replaces plain cross-entropy for writer classification.
#                      Forces embeddings into tighter, angularly-separated
#                      clusters — the single biggest metric learning improvement.
#
#   Stratified batch : 4 CEDAR + 8 Dataset2 writers per batch instead of
#                      random uniform.  Fixes the 14:1 imbalance that was
#                      under-exposing CEDAR patterns in every batch.
#
#   100 epochs       : was 50 — larger combined dataset needs more iterations.
#
# Expected epoch-1 loss: 3.0 – 6.0  (higher than before due to ArcFace scale=30)
# Loss should decrease steadily.  Below 0.1 after epoch 1 = collapse warning.
#
# Estimated time: 4 – 5 hours on a T4 GPU

import train as _train_mod
_train_mod.NUM_WORKERS = 2   # 2 workers is optimal for Colab Linux

from train import train
train(sanity=False)

In [ ]:
# ── Cell 6: Evaluate on both held-out test sets ───────────────────────────
#
# CEDAR    writers 46-55   : standard benchmark, comparable to published results
# Dataset2 writers 651-686 : second independent test, measures generalisation
#
# Primary metric: FAR @ 0.5%  (forgeries accepted at strictest banking threshold)

from evaluate import evaluate
results = evaluate(checkpoint_path='checkpoints/best.pt')

In [ ]:
# ── Cell 7: Display plots inline ──────────────────────────────────────────
from IPython.display import Image, display
import os

plot_files = [
    ('CEDAR — Score distributions',    'plots/cedar_score_dist.png'),
    ('CEDAR — FAR / FRR curve',        'plots/cedar_far_frr.png'),
    ('Dataset2 — Score distributions', 'plots/dataset2_score_dist.png'),
    ('Dataset2 — FAR / FRR curve',     'plots/dataset2_far_frr.png'),
    ('Combined — Score distributions', 'plots/combined_score_dist.png'),
    ('Combined — FAR / FRR curve',     'plots/combined_far_frr.png'),
    ('Training loss curve',            'checkpoints/loss_curve.png'),
]

for title, path in plot_files:
    if os.path.exists(path):
        print(f'── {title}')
        display(Image(path))

In [ ]:
# ── Cell 8: Operating-point table as a DataFrame ──────────────────────────
# Formats the results from Cell 6 into a tidy pandas table.
import pandas as pd
from IPython.display import display

FAR_LABELS = {'0.005': 'FAR ≤ 0.5%', '0.01': 'FAR ≤ 1%',
              '0.02':  'FAR ≤ 2%',   '0.05': 'FAR ≤ 5%'}

for r in results:
    print(f"\n── {r['name']}  (EER = {r['eer']*100:.2f}%)")
    rows = []
    for op in r['ops']:
        label = FAR_LABELS.get(str(op['far_target']), f"{op['far_target']*100:.1f}%")
        rows.append({
            'FAR target': label,
            'Threshold':  f"{op['threshold']:.4f}"       if op['threshold']   is not None else 'N/A',
            'Actual FAR': f"{op['actual_far']*100:.2f}%" if op['actual_far']  is not None else 'N/A',
            'FRR (miss)': f"{op['frr']*100:.2f}%"        if op['frr']         is not None else 'N/A',
        })
    display(pd.DataFrame(rows).set_index('FAR target'))

In [ ]:
# ── Cell 9: Download results ───────────────────────────────────────────────
# Packs checkpoints + plots into results.zip and triggers a browser download.
import zipfile, os
from google.colab import files

with zipfile.ZipFile('results.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in ('checkpoints', 'plots'):
        if not os.path.isdir(folder):
            continue
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            if os.path.isfile(fpath):
                zf.write(fpath)
                print(f'  + {fpath}')

print('\nDownloading results.zip ...')
files.download('results.zip')